In [1]:
import torch
import torchvision
from torchvision import models
from torchvision import transforms
import torch.nn as nn
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

try:
    import wandb

    wandb.init(project="convnext-tiny-codebook", job_type="train")
except ImportError:
    wandb = None
    logger.info("wandb is not available, skipping wandb.init")

In [ ]:
IMAGENET_PATH = ""

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
convnext_tiny = models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.IMAGENET1K_V1)
convnext_tiny = convnext_tiny.to(device)

In [ ]:
convnext_tiny.eval()

for param in convnext_tiny.parameters():
    param.requires_grad = False

convnext_tiny

In [ ]:
convnext_tiny_features = convnext_tiny.features
convnext_tiny_features.eval()
convnext_tiny_features

In [ ]:
def validate_epoch(
    model: nn.Module, val_dataloader: torch.utils.data.DataLoader
) -> tuple[float, float]:
    model.eval()
    top1_correct, top5_correct, total = 0, 0, 0

    with torch.no_grad():
        for inputs, labels in val_dataloader:
            inputs, labels = inputs.to(device), labels.to(device)

            outputs = model(inputs)
            _, pred = outputs.topk(5, 1, True, True)

            total += labels.size(0)
            top1_correct += (pred[:, :1] == labels.view(-1, 1)).sum().item()
            top5_correct += (pred == labels.view(-1, 1)).sum().item()

    top1_acc = (top1_correct / total) * 100
    top5_acc = (top5_correct / total) * 100
    return top1_acc, top5_acc


def train_epoch(
    model: nn.Module,
    train_dataloader: torch.utils.data.DataLoader,
    optimizer: torch.optim.Optimizer,
    criterion: nn.Module,
) -> None:
    model.train()

    for i, (images, labels) in enumerate(train_dataloader):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        output = convnext_tiny(images)
        loss = criterion(output, labels)
        loss.backward()
        optimizer.step()

        accuracy = (output.argmax(1) == labels).float().mean()

        if wandb:
            wandb.log({"Train Loss": loss.item(), "Train Accuracy": accuracy.item()})

        if i % (len(train_dataloader) // 10) == 0:
            logger.info(
                f"Iteration: {i} / {len(train_dataloader)}, Loss: {loss.item()}, Accuracy: {accuracy.item()}"
            )

In [6]:
class Codebook(nn.Module):
    def __init__(self, num_entries: int, embedding_dim: int):
        super(Codebook, self).__init__()
        self.embeddings = nn.Embedding(num_entries, embedding_dim)

        nn.init.uniform_(
            self.embeddings.weight, a=-1.0, b=1.0
        )  # Initialize codebook vectors

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # assume the input has a batch dimension
        assert len(x.shape) == 4, "Input tensor must have a batch dimension"

        # Flatten and permute to (batch, H * W, C)
        batch_size, channels, height, width = x.shape
        x_flat = x.view(batch_size, channels, height * width).permute(
            0, 2, 1
        )  # (batch, H * W, C)

        # Normalize input vectors and codebook vectors
        x_normalized = torch.functional.F.normalize(x_flat, dim=-1)  # [B, H * W, C]
        codebook_normalized = torch.functional.F.normalize(
            self.embeddings.weight, dim=-1
        )  # [num_entries, C]

        # Compute cosine similarity between input vectors and codebook vectors
        similarity = torch.matmul(
            x_normalized, codebook_normalized.t()
        )  # [B, H * W, num_entries]

        # Find the closest codebook vector for each spatial token
        indices = torch.argmax(similarity, dim=-1)  # [B, H * W]

        # Replace each token with its closest codebook vector
        quantized = self.embeddings(indices)  # [B, H * W, C]

        # Reshape back to (B, C, H, W)
        quantized = quantized.view(batch_size, height, width, channels).permute(
            0, 3, 1, 2
        )  # [B, C, H, W]

        return quantized

Test for the codebook (check if the codebook output matches the input dimensions)

In [7]:
test_codebook = Codebook(num_entries=1, embedding_dim=768)
test_codebook = test_codebook.to(device)
codebook_vector = (
    test_codebook.embeddings.weight.unsqueeze(-1).unsqueeze(-1).expand(1, 768, 7, 7)
)

t_input = torch.randn(1, 3, 224, 224).to(device)
test_output = convnext_tiny_features(t_input)
test_quantized_vectors = test_codebook(test_output)

assert test_quantized_vectors.shape == test_output.shape

# check if all vectors are close to the codebook vector
assert torch.allclose(test_quantized_vectors, codebook_vector)

In [ ]:
train_transform = transforms.Compose(
    [
        transforms.RandomResizedCrop(224),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225],
        ),
    ]
)

val_transform = transforms.Compose(
    [
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225],
        ),
    ]
)

In [ ]:
imagenet_val_ds = torchvision.datasets.ImageNet(
    root=IMAGENET_PATH, split="val", transform=val_transform
)
imagenet_train_ds = torchvision.datasets.ImageNet(
    root=IMAGENET_PATH, split="train", transform=train_transform
)

In [ ]:
BATCH_SIZE = 32
OPTIMIZER_LR = 1e-3
NUM_EPOCHS = 5

config = {
    "batch_size": BATCH_SIZE,
    "optimizer_lr": OPTIMIZER_LR,
    "num_epochs": NUM_EPOCHS,
    "optimizer": torch.optim.Adam,
}

In [ ]:
# update wandb config if it exists
if wandb:
    wandb.config.update(config)

Load Imagenet

In [ ]:
imagenet_val_dl = torch.utils.data.DataLoader(
    imagenet_val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=16,
    pin_memory=True,
)
imagenet_train_dl = torch.utils.data.DataLoader(
    imagenet_train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=16,
    pin_memory=True,
)

In [ ]:
criterion = nn.CrossEntropyLoss()

### Base accuracy results

In [ ]:
base_top1_acc, base_top5_acc = validate_epoch(convnext_tiny, imagenet_val_dl)

logger.info(f"Base top1-accuracy {base_top1_acc}, top5-accuracy {base_top5_acc}")
if wandb:
    wandb.log(
        {"Base Top1 Accuracy": base_top1_acc, "Base Top5 Accuracy": base_top5_acc}
    )

## Define the codebook and train the codebook with a random input data.

In [ ]:
codebook = Codebook(num_entries=50, embedding_dim=768)
codebook = codebook.to(device)

In [ ]:
# inject the codebook into the model after features
convnext_tiny.features.add_module("codebook", codebook)

# Set requires_grad to False for all parameters
for param in convnext_tiny.parameters():
    param.requires_grad = False

# Set requires_grad to True for the codebook parameters
for param in codebook.parameters():
    param.requires_grad = True

convnext_tiny

In [ ]:
optimizer = config["optimizer"](codebook.parameters(), lr=config["optimizer_lr"])

In [ ]:
# write a training loop to train the model
# with validation every 1/10 epochs

for epoch in range(NUM_EPOCHS):
    logger.info(f"Epoch: {epoch}")
    train_epoch(convnext_tiny, imagenet_train_dl, optimizer, criterion)

    top1_acc, top5_acc = validate_epoch(convnext_tiny, imagenet_val_dl)

    logger.info(f"Validation top1-accuracy {top1_acc}, top5-accuracy {top5_acc}")
    if wandb:
        wandb.log(
            {"Validation Top1 Accuracy": top1_acc, "Validation Top5 Accuracy": top5_acc}
        )